In [1]:
import os
import pandas as pd

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))

In [3]:
from schemas import schemas
from transformations import apply_transformations

In [4]:
import transformations

print(transformations.__file__)

C:\Users\alinaaleks\Downloads\GitHub\vessel-analytics\src\transformations.py


In [5]:
import inspect
from transformations import apply_transformations

print(inspect.getsource(apply_transformations))

def apply_transformations(df, table_name):
    """
    Apply row-level business corrections.
    These corrections fix known source data issues.
    """

    if table_name == "payments":

        # Fix payment records by payment_id
        corrections = {

            # payment_id: {
            #     "reason": "...",
            #     "changes": {
            #         column: new_value
            #     }
            # }

            781: {
                "reason": "Incorrect counterpart_id and contract_id assigned in source data",
                "changes": {
                    "counterpart_id": 233,
                    "contract_id": 501,
                }
            },

            2693: {
                "reason": "Incorrect contract_id assigned in source data",
                "changes": {
                    "contract_id": 501,
                }
            },
        }

        print(df[df["payment_id"].isin([781, 2693])])


        for payment_id, correction in corrections

In [6]:
print(schemas.keys())

dict_keys(['projects', 'vessels', 'counterparts', 'contract_types', 'contract_groups', 'contracts', 'vessel_allocation', 'payments'])


In [7]:
df = pd.read_excel(
    "../data/processed/data_cleaned_for_import.xlsx",
    sheet_name="contracts"
)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 540 entries, 0 to 539
Data columns (total 19 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   contract_id                       540 non-null    int64         
 1   counterpart_id                    540 non-null    int64         
 2   date                              540 non-null    datetime64[us]
 3   number                            540 non-null    int64         
 4   quantity_mt                       315 non-null    float64       
 5   delivery_due_date                 417 non-null    datetime64[us]
 6   total_by_ctr                      337 non-null    float64       
 7   total_fact_payments_lookup        540 non-null    float64       
 8   price_without_vat                 540 non-null    float64       
 9   vat                               540 non-null    float64       
 10  price_with_vat                    537 non-null    float64    

In [9]:
df.head(2)

,contract_id,counterpart_id,date,number,quantity_mt,delivery_due_date,total_by_ctr,total_fact_payments_lookup,price_without_vat,vat,price_with_vat,shipped_by_supplier_mt,received_on_wh_mt,contract_group_id,contract_type_id,vessel_id_loose_ctrgroups_lookup,vessel_id_ctrgroups_lookup,received_total_payments_lookup,paid_total_payments_lookup
0,1,3,2023-07-31,7429670,28600.0,2023-11-18,675052950.0,6.741610e+08,23603.25,0.0,23603.25,28562.218,28562.218,94,36,1,NaN,6.741610e+08,0.0
1,2,2,2024-01-21,7496406,1900.0,2024-02-19,81395050.0,8.151414e+07,42839.50,0.0,42839.50,1902.780,1902.780,97,36,9,9.0,8.151414e+07,0.0


In [10]:
# Проверка наличия всех листов
file = "../data/processed/data_cleaned_for_import.xlsx"

excel = pd.ExcelFile(file)

print(excel.sheet_names)

['payments', 'vessel_allocation', 'contracts', 'contract_groups', 'contract_types', 'counterparts', 'vessels', 'projects']


In [11]:
# Основной скрипт для выгрузки в CSV
input_file = "../data/processed/data_cleaned_for_import.xlsx"
output_folder = "../data/csv"

os.makedirs(output_folder, exist_ok=True)

excel = pd.ExcelFile(input_file)

for sheet in excel.sheet_names:

    print(f"Processing: {sheet}")

    df = pd.read_excel(
        excel,
        sheet_name=sheet
    )

    # Названия колонок
    df.columns = (
        df.columns
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", "_", regex=True)
    )

    # Еще раз убрать пробелы в текстовых колонках
    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns
        
    for col in text_columns:
        df[col] = (
            df[col]
                .str.strip()
                .replace("", None)
        )

    # Проверка и фильтрация колонок

    if sheet in schemas:

        expected_columns = list(schemas[sheet].keys())

        missing_columns = set(expected_columns) - set(df.columns)

        extra_columns = set(df.columns) - set(expected_columns)

        if missing_columns:
            print(
                f"WARNING: Missing columns in {sheet}: {missing_columns}"
            )

        if extra_columns:
            print(
                f"Extra columns removed from {sheet}: {extra_columns}"
            )

        df = df[
            [col for col in expected_columns if col in df.columns]
        ]

    else:
        print(f"No schema found for {sheet}")
        continue

    # Применить типы данных из схемы
    if sheet in schemas:

        for col, dtype in schemas[sheet].items():

            if col not in df.columns:
                print(f"WARNING: {col} not found in {sheet}")
                continue

            if dtype == "datetime64[ns]":

                df[col] = pd.to_datetime(
                    df[col],
                    errors="coerce"
                )

            else:
                df[col] = df[col].astype(dtype)

    else:
        print(f"Schema not found for {sheet}")

    # Применить индивидуальные исправления данных
    df = apply_transformations(
        df,
        sheet
    )

    # Проверка
    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    # Привести даты к формату для CSV
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].dt.strftime("%Y-%m-%d")

    # Сохранить в CSV
    df.to_csv(
    f"{output_folder}/{sheet}.csv",
    index=False,
    encoding="utf-8",
    sep=",",
    decimal="."
    )
    
    print(f"{sheet} saved")

Processing: payments
Extra columns removed from payments: {'received_minus_paid_calc'}
      payment_id internal_number payment_date  contract_id  counterpart_id  \
780          781       714173825   2024-01-01          233              93   
2692        2693       714914162          NaT          504             233   

      received_rub  paid_rub                  operation_type  
780            0.0   10252.0  Перечисление подотчетному лицу  
2692           NaN       NaN  Перечисление подотчетному лицу  
781 rows found: 1
Payment 781 corrected: Incorrect counterpart_id and contract_id assigned in source data
2693 rows found: 1
Payment 2693 corrected: Incorrect contract_id assigned in source data

Data types:
payment_id                  Int64
internal_number            string
payment_date       datetime64[us]
contract_id                 Int64
counterpart_id              Int64
received_rub              float64
paid_rub                  float64
operation_type             string
dtype: ob